# ShopInsight — retraduction des avis sur GPU (Google Colab)

Ce notebook retraduit les 42 687 avis Olist du portugais vers le français avec **NLLB-200** (Meta), phrase par phrase, sur le GPU gratuit de Colab.

**Avant de commencer :** menu **Exécution → Modifier le type d'exécution → GPU (T4)**, puis exécutez les cellules dans l'ordre.

Durée indicative sur un GPU T4 : environ 30 à 45 min pour `nllb-1.3b`, 15 à 25 min pour `nllb-600m` (≈ 50 000 phrases uniques). Le travail est enregistré au fur et à mesure dans votre Google Drive : si Colab se déconnecte, relancez simplement les cellules, la traduction reprend où elle s'était arrêtée.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dossier de travail dans Google Drive
Autorisez l'accès à votre Drive : le cache de traduction y est enregistré, donc rien n'est perdu en cas de déconnexion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/ShopInsight_traduction'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
!ls

## 3. Envoyer `traduction_colab.zip`
À faire une seule fois : choisissez le fichier `traduction_colab.zip` (dans le dossier du projet). Si le dossier `src` est déjà présent dans le Drive, passez cette cellule.

In [ ]:
import os, zipfile
if not os.path.exists('src/translate_reviews.py'):
    from google.colab import files
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall('.')
!ls src data/silver

## 4. Installer les bibliothèques

In [ ]:
!pip install -q transformers sentencepiece sacremoses

## 5. Choisir le modèle
`nllb-1.3b` (meilleur, plus lent), `nllb-600m` (plus rapide) ou `opus` (modèle actuel). Gardez le gagnant de la comparaison à l'aveugle.

In [ ]:
MODEL = 'nllb-1.3b'
BATCH = 64   # baissez à 32 si erreur « CUDA out of memory »

## 6. Test rapide sur 200 avis

In [ ]:
!python src/translate_reviews.py --project-dir . --model {MODEL} --batch-size {BATCH} --limit 200

## 7. Traduction complète
Le temps restant s'affiche. Si la session se coupe, réexécutez les cellules 2, 4, 5 puis celle-ci : la traduction reprend grâce au cache.

In [ ]:
!python src/translate_reviews.py --project-dir . --model {MODEL} --batch-size {BATCH} --set-default

## 8. Contrôle et téléchargement
Le zip contient la nouvelle traduction (`reviews_text_fr.jsonl.gz`), le cache et le rapport qualité.

In [ ]:
import json, glob, zipfile
rep = sorted(glob.glob('quality/translation_report*.json'), key=os.path.getmtime)[-1]
print(json.dumps(json.load(open(rep, encoding='utf-8')), ensure_ascii=False, indent=2))
out = f'resultats_traduction_{MODEL}.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['data/silver/reviews_text_fr.jsonl.gz', f'data/silver/translation_cache_fr_{MODEL}.jsonl', rep,
              rep.replace('translation_report', 'translation_sample_for_review').replace('.json', '.csv')]:
        if os.path.exists(f):
            z.write(f)
from google.colab import files
files.download(out)

## 9. Sur votre PC
1. Dans le dossier V2, renommez l'ancien fichier `data/silver/reviews_text_fr.jsonl.gz` en `reviews_text_fr_opus_V3.jsonl.gz` (sauvegarde).
2. Décompressez `resultats_traduction_….zip` à la racine du dossier V2 : il remplace `reviews_text_fr.jsonl.gz` par la nouvelle traduction.
3. Relancez `python src/evaluate_agents.py --project-dir .` : les agents utilisent automatiquement la nouvelle traduction pour leurs extraits.